### Визуализация всего поля эмбедингов

In [ ]:
!pip install transformers torch sentencepiece accelerate bitsandbytes scikit-learn pandas plotly umap-learn

In [ ]:
# %%time
# Магическая команда %%time (опционально) покажет общее время выполнения ячейки.

# ==============================================================================
# 1. ИМПОРТ БИБЛИОТЕК И НАСТРОЙКА
# ==============================================================================
import torch
from transformers import AutoModel, AutoTokenizer
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from tqdm.notebook import tqdm
import sys

# Проверяем наличие cuML и настраиваем его.
try:
    import cuml
    import cupy as cp

    print(
        f"Библиотека cuML версии {cuml.__version__} найдена. Вычисления будут на GPU."
    )
except ImportError:
    print("КРИТИЧЕСКАЯ ОШИБКА: Библиотека cuML не найдена.")
    raise

# Настраиваем рендеринг для plotly в среде Jupyter.
pio.renderers.default = "jupyterlab"
MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"

# ==============================================================================
#   <<<<<  ПАРАМЕТРЫ ВИЗУАЛИЗАЦИИ (РЕДАКТИРОВАТЬ ЗДЕСЬ)  >>>>>
# ==============================================================================
# Введите предложение, путь которого хотите отследить.
# Модель может разбить слова на части (суб-токены), вы увидите это на графике.
SENTENCE_TO_VISUALIZE = "The cat sat on the mat and looked at the moon."

# Какой процент всех точек оставить для фона?
PERCENT_TO_VIEW = 95.0


# ==============================================================================
# 2. ЗАГРУЗКА МОДЕЛИ И ТОКЕНИЗАТОРА (без изменений)
# ==============================================================================
print(f"\nЗагрузка модели и токенизатора '{MODEL_NAME}'...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_NAME, trust_remote_code=True, torch_dtype="auto", device_map="auto"
)
print("Модель и токенизатор успешно загружены.")


# ==============================================================================
# 3. ИЗВЛЕЧЕНИЕ И ПОДГОТОВКА ДАННЫХ (без изменений)
# ==============================================================================
print("\nИзвлечение и подготовка матрицы эмбеддингов...")
full_embedding_matrix = model.get_input_embeddings().weight.detach()
vocab = tokenizer.get_vocab()
vocab_size = len(vocab)
embeddings_for_viz = full_embedding_matrix[:vocab_size, :]
embeddings_tensor_float32 = embeddings_for_viz.to(torch.float32)
all_tokens = [""] * vocab_size
for token, token_id in tqdm(vocab.items(), desc="Сопоставление токенов"):
    if token_id < vocab_size:
        all_tokens[token_id] = token
print("Данные готовы.")


# ==============================================================================
# 4. СНИЖЕНИЕ РАЗМЕРНОСТИ (UMAP НА GPU) (без изменений)
# ==============================================================================
print(f"\nСнижение размерности ВСЕХ {vocab_size} токенов с помощью UMAP на GPU...")
embeddings_cupy = cp.asarray(embeddings_tensor_float32)
umap_3d = cuml.UMAP(
    n_components=3, n_neighbors=15, min_dist=0.1, metric="cosine", verbose=True
)
embeddings_3d_gpu = umap_3d.fit_transform(embeddings_cupy)
embeddings_3d_cpu = embeddings_3d_gpu.get()
print("Снижение размерности успешно завершено.")


# ==============================================================================
# 5. АВТОМАТИЧЕСКИЙ РАСЧЕТ ОПТИМАЛЬНОГО СРЕЗА (без изменений)
# ==============================================================================
print(f"\nАвтоматический расчет среза для отображения {PERCENT_TO_VIEW}% точек...")
lower_percentile = (100 - PERCENT_TO_VIEW) / 2
upper_percentile = 100 - lower_percentile
x_min, x_max = np.percentile(
    embeddings_3d_cpu[:, 0], [lower_percentile, upper_percentile]
)
y_min, y_max = np.percentile(
    embeddings_3d_cpu[:, 1], [lower_percentile, upper_percentile]
)
z_min, z_max = np.percentile(
    embeddings_3d_cpu[:, 2], [lower_percentile, upper_percentile]
)
max_len = max(x_max - x_min, y_max - y_min, z_max - z_min)
x_center, y_center, z_center = np.median(embeddings_3d_cpu, axis=0)
half_len = max_len / 2
X_RANGE = [x_center - half_len, x_center + half_len]
Y_RANGE = [y_center - half_len, y_center + half_len]
Z_RANGE = [z_center - half_len, z_center + half_len]
print("Расчет завершен.")


# ==============================================================================
# 6. ВИЗУАЛИЗАЦИЯ ПУТИ ГЕНЕРАЦИИ
# ==============================================================================
print(f"\nПодготовка данных для визуализации пути предложения...")

# --- НОВЫЙ БЛОК: ПОЛУЧАЕМ КООРДИНАТЫ ДЛЯ ПРЕДЛОЖЕНИЯ ---
path_token_ids = tokenizer.encode(SENTENCE_TO_VISUALIZE)
path_coords = embeddings_3d_cpu[path_token_ids]
path_tokens_str = [tokenizer.decode([tid]) for tid in path_token_ids]

print("Токены в предложении:")
print(" -> ".join(f"'{s}'" for s in path_tokens_str))
# ---------------------------------------------------------

# Слой 1: Фоновые точки
trace_background = go.Scatter3d(
    x=embeddings_3d_cpu[:, 0],
    y=embeddings_3d_cpu[:, 1],
    z=embeddings_3d_cpu[:, 2],
    mode="markers",
    hoverinfo="none",
    name="Все токены",
    marker=dict(size=1.5, color="lightgray", opacity=0.2),
)

# Слой 2: Линия, соединяющая точки пути
trace_path_line = go.Scatter3d(
    x=path_coords[:, 0],
    y=path_coords[:, 1],
    z=path_coords[:, 2],
    mode="lines",
    hoverinfo="none",
    name="Путь генерации",
    line=dict(color="yellow", width=5),
)

# Слой 3: Сами точки пути (крупные и с подсказками)
trace_path_points = go.Scatter3d(
    x=path_coords[:, 0],
    y=path_coords[:, 1],
    z=path_coords[:, 2],
    mode="markers",
    text=path_tokens_str,
    name="Токены предложения",
    hovertemplate=(
        "<b>Токен:</b> %{text}<br><br>"
        + "<b>Координаты:</b><br>"
        + "X: %{x:.3f}<br>Y: %{y:.3f}<br>Z: %{z:.3f}"
        + "<extra></extra>"
    ),
    marker=dict(size=7, color="red", opacity=1.0, line=dict(width=1, color="white")),
)

# Собираем все слои в один график. Порядок важен для наложения.
fig = go.Figure(data=[trace_background, trace_path_line, trace_path_points])

fig.update_layout(
    title=f"Путь генерации предложения в пространстве эмбеддингов",
    margin=dict(l=0, r=0, b=0, t=40),
    height=800,
    scene=dict(
        xaxis=dict(range=X_RANGE, title="Компонента 1"),
        yaxis=dict(range=Y_RANGE, title="Компонента 2"),
        zaxis=dict(range=Z_RANGE, title="Компонента 3"),
        aspectmode="cube",
        bgcolor="rgb(20, 24, 54)",
    ),
    hoverlabel=dict(bgcolor="white", font_size=14),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
)

fig.show()

### Визуализация пути генерации предложения

In [1]:
# %%time
# Магическая команда %%time (опционально) покажет общее время выполнения ячейки.

# ==============================================================================
# 1. ИМПОРТ БИБЛИОТЕК
# ==============================================================================
import torch
from transformers import AutoModel, AutoTokenizer
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from tqdm.notebook import tqdm
import ipywidgets as widgets
from IPython.display import display

# Проверяем наличие cuML и настраиваем его.
try:
    import cuml
    import cupy as cp

    print(
        f"Библиотека cuML версии {cuml.__version__} найдена. Вычисления будут на GPU."
    )
except ImportError:
    print("КРИТИЧЕСКАЯ ОШИБКА: Библиотека cuML не найдена. Установите RAPIDS AI.")
    raise

# Настраиваем рендеринг для plotly в среде Jupyter.
pio.renderers.default = "jupyterlab"
MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"

# ==============================================================================
#   <<<<<  КОНФИГУРАЦИЯ ВИЗУАЛИЗАЦИИ (РЕДАКТИРОВАТЬ ЗДЕСЬ)  >>>>>
# ==============================================================================
ALGORITHM_TO_USE = "PCA"
SENTENCE_TO_VISUALIZE = "The cat sat on the mat and looked at the moon."
PERCENT_TO_VIEW = 99.9
RANDOM_SEED = 42
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
N_NEIGHBORS_TO_SHOW = 100

print(f"Выбран алгоритм снижения размерности: {ALGORITHM_TO_USE}")

# ==============================================================================
# 2. ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ (можно закомментировать после 1-го запуска)
# ==============================================================================
print(f"\nЗагрузка модели и токенизатора '{MODEL_NAME}'...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_NAME, trust_remote_code=True, torch_dtype="auto", device_map="auto"
)
print("Модель и токенизатор успешно загружены.")

print("\nИзвлечение и подготовка матрицы эмбеддингов...")
full_embedding_matrix = model.get_input_embeddings().weight.detach()
vocab = tokenizer.get_vocab()
vocab_size = len(vocab)
embeddings_tensor_float32 = full_embedding_matrix[:vocab_size, :].to(torch.float32)
embeddings_cupy = cp.asarray(embeddings_tensor_float32)

all_tokens = [""] * vocab_size
for token, token_id in tqdm(vocab.items(), desc="Сопоставление токенов"):
    if token_id < vocab_size:
        all_tokens[token_id] = token
print("Данные готовы.")

# ==============================================================================
# 3. ПРЕДВАРИТЕЛЬНЫЙ РАСЧЕТ БЛИЖАЙШИХ СОСЕДЕЙ (НА GPU)
# ==============================================================================
print(f"\nПредварительный расчет {N_NEIGHBORS_TO_SHOW} ближайших соседей...")
nn_model = cuml.neighbors.NearestNeighbors(
    n_neighbors=N_NEIGHBORS_TO_SHOW + 1, metric="cosine"
)
nn_model.fit(embeddings_cupy)
_, nearest_neighbors_indices_gpu = nn_model.kneighbors(embeddings_cupy)
nearest_neighbors_indices_cpu = nearest_neighbors_indices_gpu.get()
print("Расчет соседей завершен.")

# ==============================================================================
# 4. СНИЖЕНИЕ РАЗМЕРНОСТИ
# ==============================================================================
print(
    f"\nСнижение размерности {vocab_size} токенов с помощью {ALGORITHM_TO_USE} на GPU..."
)
match ALGORITHM_TO_USE:
    case "UMAP":
        reducer = cuml.UMAP(
            n_components=3,
            n_neighbors=UMAP_N_NEIGHBORS,
            min_dist=UMAP_MIN_DIST,
            metric="cosine",
            random_state=RANDOM_SEED,
            verbose=True,
        )
    case "PCA":
        reducer = cuml.PCA(n_components=3)
    case "t-SNE":
        reducer = cuml.TSNE(
            n_components=3, metric="cosine", random_state=RANDOM_SEED, verbose=True
        )
    case _:
        raise ValueError(f"Неизвестный алгоритм: '{ALGORITHM_TO_USE}'.")
embeddings_3d_gpu = reducer.fit_transform(embeddings_cupy)
embeddings_3d_cpu = embeddings_3d_gpu.get()
print("Снижение размерности завершено.")

# ==============================================================================
# 5. АВТОМАТИЧЕСКИЙ РАСЧЕТ ОПТИМАЛЬНОГО СРЕЗА
# ==============================================================================
print(f"\nАвтоматический расчет среза для отображения {PERCENT_TO_VIEW}% точек...")
lower_percentile = (100 - PERCENT_TO_VIEW) / 2
upper_percentile = 100 - lower_percentile
x_min, x_max = np.percentile(
    embeddings_3d_cpu[:, 0], [lower_percentile, upper_percentile]
)
y_min, y_max = np.percentile(
    embeddings_3d_cpu[:, 1], [lower_percentile, upper_percentile]
)
z_min, z_max = np.percentile(
    embeddings_3d_cpu[:, 2], [lower_percentile, upper_percentile]
)
max_len = max(x_max - x_min, y_max - y_min, z_max - z_min)
x_center, y_center, z_center = np.median(embeddings_3d_cpu, axis=0)
half_len = max_len / 2
X_RANGE, Y_RANGE, Z_RANGE = (
    [x_center - half_len, x_center + half_len],
    [y_center - half_len, y_center + half_len],
    [z_center - half_len, z_center + half_len],
)
print("Расчет завершен.")

# ==============================================================================
# 6. ИНТЕРАКТИВНАЯ ВИЗУАЛИЗАЦИЯ С ПАНЕЛЬЮ СОСЕДЕЙ
# ==============================================================================
print(f"\nСоздание интерактивной визуализации для {ALGORITHM_TO_USE}...")

# --- Подготовка данных для пути ---
path_token_ids = tokenizer.encode(SENTENCE_TO_VISUALIZE)
path_coords = embeddings_3d_cpu[path_token_ids]
path_tokens_str = [tokenizer.decode([tid]) for tid in path_token_ids]
path_numbers = [str(i + 1) for i in range(len(path_token_ids))]
custom_data_for_path = np.stack((path_tokens_str, path_numbers), axis=-1)

# --- Создание "рецептов" для слоев графика ---
trace_all_tokens = go.Scatter3d(
    x=embeddings_3d_cpu[:, 0],
    y=embeddings_3d_cpu[:, 1],
    z=embeddings_3d_cpu[:, 2],
    mode="markers",
    hoverinfo="none",
    marker=dict(size=2, color="lightblue", opacity=0.5),
)
trace_path_line = go.Scatter3d(
    x=path_coords[:, 0],
    y=path_coords[:, 1],
    z=path_coords[:, 2],
    mode="lines",
    hoverinfo="none",
    name="Путь генерации",
    line=dict(color="lime", width=4),
)
trace_path_points = go.Scatter3d(
    x=path_coords[:, 0],
    y=path_coords[:, 1],
    z=path_coords[:, 2],
    mode="markers+text",
    text=path_numbers,
    customdata=custom_data_for_path,
    name="Токены предложения (клик/наведение)",
    hoverinfo="none",
    textfont=dict(size=11, color="white"),
    textposition="middle center",
    marker=dict(size=8, color="red", opacity=1.0, line=dict(width=2, color="white")),
)

# --- Создаем виджеты: график и панель для соседей ---
fig = go.FigureWidget(data=[trace_all_tokens, trace_path_line, trace_path_points])
neighbor_list_widget = widgets.HTML(
    value="<i>Наведите курсор на любую точку, чтобы увидеть ее ближайших семантических соседей.</i>",
    # ИЗМЕНЕНО: Задаем МАКСИМАЛЬНУЮ высоту и АВТОМАТИЧЕСКУЮ прокрутку
    layout=widgets.Layout(
        width="250px",
        max_height="800px",  # Задает потолок высоты, равный высоте графика
        border="solid 1px lightgray",
        padding="10px",
        overflow_y="auto",  # Добавляет прокрутку только когда контент не помещается
    ),
)
fig.update_layout(
    title=f"Пространство эмбеддингов ({ALGORITHM_TO_USE})",
    margin=dict(l=0, r=0, b=0, t=40),
    height=800,
    scene=dict(
        xaxis=dict(range=X_RANGE, title="Компонента 1"),
        yaxis=dict(range=Y_RANGE, title="Компонента 2"),
        zaxis=dict(range=Z_RANGE, title="Компонента 3"),
        aspectmode="cube",
        bgcolor="rgb(20, 24, 54)",
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.5)),
    ),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
)


# --- Функции-обработчики событий ---
def update_neighbor_list(global_token_id):
    hovered_token = (
        all_tokens[global_token_id].replace("<", "&lt;").replace(">", "&gt;")
    )
    neighbor_indices = nearest_neighbors_indices_cpu[global_token_id][1:]
    html_content = f"<h4>Ближайшие соседи для<br><b>'{hovered_token}'</b>:</h4><ol>"
    for idx in neighbor_indices:
        neighbor_token = all_tokens[idx].replace("<", "&lt;").replace(">", "&gt;")
        html_content += f"<li>{neighbor_token}</li>"
    html_content += "</ol>"
    neighbor_list_widget.value = html_content


def handle_background_hover(trace, points, state):
    if not points.point_inds:
        return
    update_neighbor_list(points.point_inds[0])


def handle_path_hover(trace, points, state):
    if not points.point_inds:
        return
    update_neighbor_list(path_token_ids[points.point_inds[0]])


def handle_path_click(trace, points, state):
    if not points.point_inds:
        return
    point_index = points.point_inds[0]
    target_x, target_y, target_z = (
        trace.x[point_index],
        trace.y[point_index],
        trace.z[point_index],
    )
    token_name = trace.customdata[point_index][0]
    print(f"Приближение к точке {point_index + 1}: токен '{token_name}'...")
    with fig.batch_update():
        fig.update_layout(
            scene_camera=dict(
                center=dict(x=target_x, y=target_y, z=target_z),
                eye=dict(x=target_x + 0.1, y=target_y + 0.1, z=target_z + 0.1),
                up=dict(x=0, y=0, z=1),
            )
        )


# --- Привязываем обработчики к "живым" слоям внутри FigureWidget ---
fig.data[0].on_hover(handle_background_hover)
fig.data[2].on_hover(handle_path_hover)
fig.data[2].on_click(handle_path_click)

# --- Отображаем итоговый макет ---
app_layout = widgets.HBox([fig, neighbor_list_widget])
print("\nГрафик готов. Можно взаимодействовать.")
display(app_layout)

Библиотека cuML версии 25.06.00 найдена. Вычисления будут на GPU.
Выбран алгоритм снижения размерности: PCA

Загрузка модели и токенизатора 'Qwen/Qwen2-1.5B-Instruct'...
Модель и токенизатор успешно загружены.

Извлечение и подготовка матрицы эмбеддингов...


Сопоставление токенов:   0%|          | 0/151646 [00:00<?, ?it/s]

Данные готовы.

Предварительный расчет 100 ближайших соседей...
Расчет соседей завершен.

Снижение размерности 151646 токенов с помощью PCA на GPU...
Снижение размерности завершено.

Автоматический расчет среза для отображения 99.9% точек...
Расчет завершен.

Создание интерактивной визуализации для PCA...

График готов. Можно взаимодействовать.


    'data': [{'hoverinfo': 'none',
              'marker': {'color': 'lightblue'…